In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import leidenalg
import squidpy as sq
from scipy import sparse
from matplotlib.colors import ListedColormap
import os
import numpy as np
import pandas as pd
from scipy.spatial import KDTree
from tqdm import tqdm
import seaborn as sns
import geopandas as gpd
from scipy.io import mmread
from scipy.spatial.distance import pdist
from scipy.stats import entropy

In [ ]:
def compute_tumor_neighbors(adata, cell_type_col="cell_type", spatial_key="spatial", radius=500):
    if spatial_key not in adata.obsm:
        raise ValueError(f"Spatial key '{spatial_key}' not found in `adata.obsm`.")
    if cell_type_col not in adata.obs:
        raise ValueError(f"Cell type column '{cell_type_col}' not found in `adata.obs`.")

    coords = adata.obsm[spatial_key]
    cell_types = adata.obs[cell_type_col]

    tumor_mask = cell_types.str.startswith("MP")
    non_malignant_mask = ~tumor_mask

    tumor_coords = coords[tumor_mask]
    non_malignant_coords = coords[non_malignant_mask]

    tumor_subtypes = cell_types[tumor_mask].values
    unique_tumor_subtypes = np.unique(tumor_subtypes)

    tumor_tree = KDTree(tumor_coords)

    tumor_neighbor_counts = {t: [] for t in unique_tumor_subtypes}
    
    for i in tqdm(range(len(non_malignant_coords)), desc="Computing tumor neighborhood composition"):
        neighbors_idx = tumor_tree.query_ball_point(non_malignant_coords[i], radius)
        neighbor_subtypes = tumor_subtypes[neighbors_idx]
        subtype_counts = {t: np.sum(neighbor_subtypes == t) for t in unique_tumor_subtypes}
        
        for t in unique_tumor_subtypes:
            tumor_neighbor_counts[t].append(subtype_counts.get(t, 0))

    tumor_neighbor_df = pd.DataFrame(tumor_neighbor_counts)
    tumor_neighbor_df.index = adata.obs.loc[non_malignant_mask].index
    
    return tumor_neighbor_df

def permutation_testing(adata, radius, cell_type_col="cell_type", num_permutations=100):
    tumor_mask = adata.obs[cell_type_col].str.startswith("MP")
    original_tumor_labels = adata.obs.loc[tumor_mask, cell_type_col].copy()

    permuted_counts = []

    for _ in tqdm(range(num_permutations), desc="Performing permutation testing"):
        shuffled_tumor_labels = np.random.permutation(original_tumor_labels.values)
        adata.obs.loc[tumor_mask, cell_type_col] = shuffled_tumor_labels

        shuffled_neighbors = compute_tumor_neighbors(adata, cell_type_col=cell_type_col, radius=radius)
        permuted_counts.append(shuffled_neighbors)

    adata.obs.loc[tumor_mask, cell_type_col] = original_tumor_labels

    all_permuted = pd.concat(permuted_counts, keys=range(num_permutations))
    mean_permuted_counts = all_permuted.groupby(level=1).mean()
    std_permuted_counts = all_permuted.groupby(level=1).std()

    return mean_permuted_counts, std_permuted_counts, permuted_counts

def compute_z_scores_per_cell_type(adata, tumor_neighbor_df, mean_permuted_counts, std_permuted_counts, cell_type_col="cell_type"):
    tumor_mask = adata.obs[cell_type_col].str.startswith("MP")
    non_malignant_mask = ~tumor_mask

    observed_means = tumor_neighbor_df.groupby(adata.obs.loc[non_malignant_mask, cell_type_col]).mean()
    expected_means = mean_permuted_counts.groupby(adata.obs.loc[non_malignant_mask, cell_type_col]).mean()
    expected_stds = std_permuted_counts.groupby(adata.obs.loc[non_malignant_mask, cell_type_col]).mean()

    z_scores = (observed_means - expected_means) / expected_stds
    z_scores = z_scores.loc[~z_scores.index.str.startswith("MP")]

    return z_scores

def compute_p_values_per_cell_type(adata, tumor_neighbor_df, permuted_counts, cell_type_col="cell_type"):
    tumor_mask = adata.obs[cell_type_col].str.startswith("MP")
    non_malignant_mask = ~tumor_mask
    non_malignant_cell_types = adata.obs.loc[non_malignant_mask, cell_type_col]

    observed_means = tumor_neighbor_df.groupby(non_malignant_cell_types).mean()

    permuted_means = [
        df.groupby(non_malignant_cell_types).mean()
        for df in permuted_counts
    ]

    p_values = observed_means.copy()
    
    for tumor_type in observed_means.columns:
        observed = observed_means[tumor_type]
        permuted = np.array([df[tumor_type] for df in permuted_means])
        expected = np.mean(permuted, axis=0)

        diffs = np.abs(permuted - expected) 
        obs_diffs = np.abs(observed.values - expected)

        p = np.mean(diffs >= obs_diffs, axis=0)
        p_values[tumor_type] = p

    return p_values

def get_significance_stars(pval):
        if pval < 0.001:
            return '***'
        elif pval < 0.01:
            return '**'
        elif pval < 0.05:
            return '*'
        else:
            return ''

In [ ]:
#set the working directory
os.chdir('/home/fceccarelli/home3/OT_simulation/code/VisiumHD/segmentation')
os.makedirs("res", exist_ok=True)

#read all folders in the directory
folders = [x for x in os.listdir() if os.path.isdir(x) and x.startswith("Pat")]
folders.sort()

radius = 500
um = "008"
z_scores_list = []
p_values_list = []

for folder in folders:
    print(f"Processing patient: {folder}")

    sp_adata = anndata.read_h5ad(folder + "/" + folder + "_" + um + "_programs.h5ad")
    #sp_adata.obs["cell_type"] = sp_adata.obs["cell_type"].astype(str).replace(regex=r"^T cells.*", value="T cells")
    
    tumor_neighbor_df = compute_tumor_neighbors(sp_adata, radius=radius)
    mean_permuted_counts, std_permuted_counts, permutations = permutation_testing(sp_adata, radius=radius)
    
    z_scores = compute_z_scores_per_cell_type(sp_adata, tumor_neighbor_df, mean_permuted_counts, std_permuted_counts)
    p_values = compute_p_values_per_cell_type(sp_adata, tumor_neighbor_df, permutations)

    z_scores.to_csv(f"res/{folder}_z_scores.csv")
    p_values.to_csv(f"res/{folder}_p_values.csv")
    
    # Create combined annotation with Z-scores and significance stars
    annot = pd.DataFrame('', index=z_scores.index, columns=z_scores.columns)

    for row in z_scores.index:
        for col in z_scores.columns:
            z = z_scores.loc[row, col]
            p = p_values.loc[row, col]
            annot.loc[row, col] = f"{z:.2f}{get_significance_stars(p)}"

    plt.figure(figsize=(10, 6))
    sns.heatmap(z_scores, annot=annot, fmt='', cmap="coolwarm", center=0, linewidths=0.5)
    plt.title(f"Tumor-Normal Interactions (radius = {radius}) \n{folder}")
    plt.xlabel("Tumor Subtypes")
    plt.ylabel("Non-Malignant Cell Types")
    plt.tight_layout()
    plt.show()    